# Matchup Recommendation Dataset

## Objective

This notebook prepares the data layer for a matchup-based lineup recommender.

The goal is to support a dashboard workflow where a coach or analyst can:

- select their team
- select an opponent
- select or define the opponent lineup
- receive recommended lineups based on historical performance, player impact, matchup strength, and reliability

This notebook does not build the app interface yet. Instead, it creates clean datasets that the Streamlit app can load directly.

In [17]:
import pandas as pd
import numpy as np
import ast

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 200)

## Load required datasets

We load the validated stint dataset and processed metric outputs generated in previous notebooks.

In [18]:
stints_df = pd.read_pickle("../data/interim/stints_multigame_validated.pkl")

player_impact = pd.read_csv("../data/processed/player_impact.csv")
reliable_player_impact = pd.read_csv("../data/processed/reliable_player_impact.csv")
lineup_perf = pd.read_csv("../data/processed/lineup_performance.csv")
reliable_lineups = pd.read_csv("../data/processed/reliable_lineups.csv")
pair_perf = pd.read_csv("../data/processed/reliable_pair_performance.csv")

stints_df.shape, player_impact.shape, lineup_perf.shape

((6293, 22), (383, 6), (3794, 8))

## Build player lookup table

We create a clean player lookup table containing player IDs and readable player names.

In [19]:
player_name_records = []

for _, row in stints_df.iterrows():
    for pid, pname in zip(row["home_lineup"], row["home_lineup_names"]):
        player_name_records.append({
            "player_id": str(pid),
            "player_name": pname
        })

    for pid, pname in zip(row["away_lineup"], row["away_lineup_names"]):
        player_name_records.append({
            "player_id": str(pid),
            "player_name": pname
        })

player_lookup = (
    pd.DataFrame(player_name_records)
    .drop_duplicates()
    .sort_values("player_name")
    .reset_index(drop=True)
)

player_lookup.head()

,player_id,player_name
0,920,A.C. Green
1,2062,A.J. Guyton
2,243,Aaron McKie
3,1425,Aaron Williams
4,228,Adam Keefe


## Build team matchup table

We create a readable team reference table from the validated stint data.

Team IDs are mapped to official NBA team names so the dashboard can present user-friendly dropdowns instead of numeric identifiers.

In [20]:
# =========================================================
# NBA team lookup table
# =========================================================

nba_team_lookup = {
    "1610612737": "Atlanta Hawks",
    "1610612738": "Boston Celtics",
    "1610612751": "Brooklyn Nets",
    "1610612766": "Charlotte Hornets",
    "1610612741": "Chicago Bulls",
    "1610612739": "Cleveland Cavaliers",
    "1610612742": "Dallas Mavericks",
    "1610612743": "Denver Nuggets",
    "1610612765": "Detroit Pistons",
    "1610612744": "Golden State Warriors",
    "1610612745": "Houston Rockets",
    "1610612754": "Indiana Pacers",
    "1610612746": "Los Angeles Clippers",
    "1610612747": "Los Angeles Lakers",
    "1610612763": "Memphis Grizzlies",
    "1610612748": "Miami Heat",
    "1610612749": "Milwaukee Bucks",
    "1610612750": "Minnesota Timberwolves",
    "1610612740": "New Orleans Pelicans",
    "1610612752": "New York Knicks",
    "1610612760": "Oklahoma City Thunder",
    "1610612753": "Orlando Magic",
    "1610612755": "Philadelphia 76ers",
    "1610612756": "Phoenix Suns",
    "1610612757": "Portland Trail Blazers",
    "1610612758": "Sacramento Kings",
    "1610612759": "San Antonio Spurs",
    "1610612761": "Toronto Raptors",
    "1610612762": "Utah Jazz",
    "1610612764": "Washington Wizards"
}

# Build clean teams table

home_teams = stints_df[["home_team_id"]].rename(
    columns={"home_team_id": "team_id"}
)

away_teams = stints_df[["away_team_id"]].rename(
    columns={"away_team_id": "team_id"}
)

teams = (
    pd.concat([home_teams, away_teams], axis=0)
    .drop_duplicates()
    .reset_index(drop=True)
)

teams["team_id"] = teams["team_id"].astype(str)

teams["team_name"] = teams["team_id"].map(nba_team_lookup)

teams["team_name"] = teams["team_name"].fillna(
    "Unknown Team " + teams["team_id"].astype(str)
)

teams = teams.sort_values("team_name").reset_index(drop=True)

teams.head()

,team_id,team_name
0,1610612737,Atlanta Hawks
1,1610612738,Boston Celtics
2,1610612751,Brooklyn Nets
3,1610612766,Charlotte Hornets
4,1610612741,Chicago Bulls


##  Build team-player roster table

For the recommender, the app needs to know which players belong to each team in the processed sample.

This roster is inferred from reconstructed stints and enriched with readable team names.

In [21]:

roster_records = []

for _, row in stints_df.iterrows():

    # Home team players
    for pid, pname in zip(row["home_lineup"], row["home_lineup_names"]):
        roster_records.append({
            "team_id": str(row["home_team_id"]),
            "player_id": str(pid),
            "player_name": pname
        })

    # Away team players
    for pid, pname in zip(row["away_lineup"], row["away_lineup_names"]):
        roster_records.append({
            "team_id": str(row["away_team_id"]),
            "player_id": str(pid),
            "player_name": pname
        })

team_rosters = (
    pd.DataFrame(roster_records)
    .drop_duplicates()
    .reset_index(drop=True)
)

team_rosters["team_id"] = team_rosters["team_id"].astype(str)

team_rosters = team_rosters.merge(
    teams[["team_id", "team_name"]],
    on="team_id",
    how="left"
)

team_rosters = (
    team_rosters
    .sort_values(["team_name", "player_name"])
    .reset_index(drop=True)
)

team_rosters.head()

,team_id,player_id,player_name,team_name
0,1610612737,673,Alan Henderson,Atlanta Hawks
1,1610612737,1533,Anthony Johnson,Atlanta Hawks
2,1610612737,292,Anthony Miller,Atlanta Hawks
3,1610612737,1898,Cal Bowdler,Atlanta Hawks
4,1610612737,1544,Chris Crawford,Atlanta Hawks


##  Parse lineup fields

Lineup performance datasets store lineup identifiers as text after CSV export. We parse them back into list format.

In [22]:
def parse_lineup(lineup_value):
    if isinstance(lineup_value, list):
        return lineup_value

    if isinstance(lineup_value, tuple):
        return list(lineup_value)

    try:
        parsed = ast.literal_eval(str(lineup_value))
        return list(parsed)
    except:
        return []


lineup_perf = lineup_perf.copy()

lineup_perf["home_lineup_parsed"] = lineup_perf["home_lineup_tuple"].apply(parse_lineup)
lineup_perf["away_lineup_parsed"] = lineup_perf["away_lineup_tuple"].apply(parse_lineup)

lineup_perf[["home_lineup_tuple", "home_lineup_parsed"]].head()

,home_lineup_tuple,home_lineup_parsed
0,"('1000', '1005', '1508', '1509', '165')","[1000, 1005, 1508, 1509, 165]"
1,"('1000', '1005', '1508', '1509', '165')","[1000, 1005, 1508, 1509, 165]"
2,"('1000', '1005', '1508', '165', '1749')","[1000, 1005, 1508, 165, 1749]"
3,"('1000', '1005', '1508', '165', '1749')","[1000, 1005, 1508, 165, 1749]"
4,"('1000', '1005', '1508', '165', '1749')","[1000, 1005, 1508, 165, 1749]"


## Add readable lineup names

We convert player IDs inside each lineup into readable names for dashboard display.

In [23]:
player_name_lookup = dict(
    zip(player_lookup["player_id"], player_lookup["player_name"])
)

def lineup_to_names(lineup):
    return [
        player_name_lookup.get(str(pid), str(pid))
        for pid in lineup
    ]

lineup_perf["home_lineup_names"] = lineup_perf["home_lineup_parsed"].apply(lineup_to_names)
lineup_perf["away_lineup_names"] = lineup_perf["away_lineup_parsed"].apply(lineup_to_names)

lineup_perf[["home_lineup_names", "away_lineup_names"]].head()

,home_lineup_names,away_lineup_names
0,"[Shandon Anderson, Walt Williams, Maurice Taylor, Kelvin Cato, Hakeem Olajuwon]","[Chauncey Billups, Rasho Nesterovic, Wally Szczerbiak, Terrell Brandon, Anthony Peeler]"
1,"[Shandon Anderson, Walt Williams, Maurice Taylor, Kelvin Cato, Hakeem Olajuwon]","[Chauncey Billups, Rasho Nesterovic, Terrell Brandon, Anthony Peeler, Sam Mitchell]"
2,"[Shandon Anderson, Walt Williams, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley]","[Darvin Ham, Tim Thomas, Sam Cassell, Scott Williams, Lindsey Hunter]"
3,"[Shandon Anderson, Walt Williams, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley]","[Vlade Divac, Lawrence Funderburke, Scot Pollard, Jason Williams, Jon Barry]"
4,"[Shandon Anderson, Walt Williams, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley]","[Vlade Divac, Scot Pollard, Jason Williams, Chris Webber, Doug Christie]"


## Build candidate lineup table

The recommender should only recommend lineups that have actually appeared in the historical sample.

This avoids suggesting unrealistic or untested combinations in the first version of the app.

In [24]:
candidate_home = lineup_perf[[
    "home_lineup_tuple",
    "home_lineup_names",
    "total_home_points",
    "total_away_points",
    "total_duration",
    "total_stints",
    "net_points",
    "net_per_60"
]].copy()

candidate_home = candidate_home.rename(
    columns={
        "home_lineup_tuple": "lineup_ids",
        "home_lineup_names": "lineup_names",
        "total_home_points": "points_for",
        "total_away_points": "points_against"
    }
)

candidate_home["lineup_side"] = "home"

candidate_lineups = candidate_home.copy()

candidate_lineups["lineup_display"] = candidate_lineups["lineup_names"].apply(
    lambda names: " | ".join(names)
)

candidate_lineups = candidate_lineups.sort_values(
    ["net_per_60", "total_duration"],
    ascending=[False, False]
).reset_index(drop=True)

candidate_lineups.head()

,lineup_ids,lineup_names,points_for,points_against,total_duration,total_stints,net_points,net_per_60,lineup_side,lineup_display
0,"('1088', '146', '376', '688', '693')","[Chucky Atkins, Jud Buechler, Eric Montross, Michael Curry, Joe Smith]",3,0,3,1,3,60.000000,home,Chucky Atkins | Jud Buechler | Eric Montross | Michael Curry | Joe Smith
1,"('1000', '1508', '165', '1749', '672')","[Shandon Anderson, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley, Matt Bullard]",2,0,2,1,2,60.000000,home,Shandon Anderson | Maurice Taylor | Hakeem Olajuwon | Cuttino Mobley | Matt Bullard
2,"('1732', '1888', '393', '436', '63')","[Felipe Lopez, Richard Hamilton, Rod Strickland, Juwan Howard, Michael Smith]",2,0,2,1,2,60.000000,home,Felipe Lopez | Richard Hamilton | Rod Strickland | Juwan Howard | Michael Smith
3,"('1717', '363', '458', '919', '93')","[Dirk Nowitzki, Christian Laettner, Howard Eisley, Loy Vaught, Hubert Davis]",6,0,7,1,6,51.428571,home,Dirk Nowitzki | Christian Laettner | Howard Eisley | Loy Vaught | Hubert Davis
4,"('1499', '1559', '179', '2040', '677')","[Tony Battie, Adrian Griffin, Bryant Stith, Jerome Moiso, Eric Williams]",5,0,6,1,5,50.000000,home,Tony Battie | Adrian Griffin | Bryant Stith | Jerome Moiso | Eric Williams


##  Add lineup team ownership

A lineup can be associated with a team by checking which team used that lineup in the validated stint data.

In [26]:
candidate_home = lineup_perf[[
    "home_lineup_tuple",
    "home_lineup_names",
    "total_home_points",
    "total_away_points",
    "total_duration",
    "total_stints",
    "net_points",
    "net_per_60"
]].copy()

candidate_home = candidate_home.rename(
    columns={
        "home_lineup_tuple": "lineup_ids",
        "home_lineup_names": "lineup_names",
        "total_home_points": "points_for",
        "total_away_points": "points_against"
    }
)

candidate_home["lineup_side"] = "home"

candidate_lineups = candidate_home.copy()

candidate_lineups["lineup_display"] = candidate_lineups["lineup_names"].apply(
    lambda names: " | ".join(names)
)

candidate_lineups = candidate_lineups.sort_values(
    ["net_per_60", "total_duration"],
    ascending=[False, False]
).reset_index(drop=True)

# Convert lineup IDs to tuple for merging
candidate_lineups["lineup_ids_tuple"] = candidate_lineups["lineup_ids"].apply(
    lambda x: tuple(parse_lineup(x))
)

# Build lineup-to-team lookup from validated stints
lineup_team_records = []

for _, row in stints_df.iterrows():
    lineup_team_records.append({
        "team_id": str(row["home_team_id"]),
        "lineup_ids_tuple": tuple(row["home_lineup"])
    })

    lineup_team_records.append({
        "team_id": str(row["away_team_id"]),
        "lineup_ids_tuple": tuple(row["away_lineup"])
    })

lineup_team_lookup_df = (
    pd.DataFrame(lineup_team_records)
    .drop_duplicates()
)

# Add team ownership
candidate_lineups = candidate_lineups.merge(
    lineup_team_lookup_df,
    on="lineup_ids_tuple",
    how="left"
)

# Add readable team names
candidate_lineups["team_id"] = candidate_lineups["team_id"].astype(str)

candidate_lineups = candidate_lineups.merge(
    teams[["team_id", "team_name"]],
    on="team_id",
    how="left"
)

candidate_lineups.head()

,lineup_ids,lineup_names,points_for,points_against,total_duration,total_stints,net_points,net_per_60,lineup_side,lineup_display,lineup_ids_tuple,team_id,team_name
0,"('1088', '146', '376', '688', '693')","[Chucky Atkins, Jud Buechler, Eric Montross, Michael Curry, Joe Smith]",3,0,3,1,3,60.000000,home,Chucky Atkins | Jud Buechler | Eric Montross | Michael Curry | Joe Smith,"(1088, 146, 376, 688, 693)",1610612765,Detroit Pistons
1,"('1000', '1508', '165', '1749', '672')","[Shandon Anderson, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley, Matt Bullard]",2,0,2,1,2,60.000000,home,Shandon Anderson | Maurice Taylor | Hakeem Olajuwon | Cuttino Mobley | Matt Bullard,"(1000, 1508, 165, 1749, 672)",1610612745,Houston Rockets
2,"('1732', '1888', '393', '436', '63')","[Felipe Lopez, Richard Hamilton, Rod Strickland, Juwan Howard, Michael Smith]",2,0,2,1,2,60.000000,home,Felipe Lopez | Richard Hamilton | Rod Strickland | Juwan Howard | Michael Smith,"(1732, 1888, 393, 436, 63)",1610612764,Washington Wizards
3,"('1717', '363', '458', '919', '93')","[Dirk Nowitzki, Christian Laettner, Howard Eisley, Loy Vaught, Hubert Davis]",6,0,7,1,6,51.428571,home,Dirk Nowitzki | Christian Laettner | Howard Eisley | Loy Vaught | Hubert Davis,"(1717, 363, 458, 919, 93)",1610612742,Dallas Mavericks
4,"('1499', '1559', '179', '2040', '677')","[Tony Battie, Adrian Griffin, Bryant Stith, Jerome Moiso, Eric Williams]",5,0,6,1,5,50.000000,home,Tony Battie | Adrian Griffin | Bryant Stith | Jerome Moiso | Eric Williams,"(1499, 1559, 179, 2040, 677)",1610612738,Boston Celtics


##  Create player impact lookup for recommendation scoring

Player impact values are used to estimate lineup strength and opponent strength.

In [27]:
player_impact["player_id"] = player_impact["player_id"].astype(str)

player_impact_lookup = dict(
    zip(player_impact["player_id"], player_impact["impact_per_60_sec"])
)

def avg_lineup_impact(lineup):
    impacts = [
        player_impact_lookup.get(str(pid), 0)
        for pid in lineup
    ]

    if len(impacts) == 0:
        return 0

    return np.mean(impacts)

candidate_lineups["avg_player_impact"] = candidate_lineups["lineup_ids_tuple"].apply(
    avg_lineup_impact
)

candidate_lineups.head()

,lineup_ids,lineup_names,points_for,points_against,total_duration,total_stints,net_points,net_per_60,lineup_side,lineup_display,lineup_ids_tuple,team_id,team_name,avg_player_impact
0,"('1088', '146', '376', '688', '693')","[Chucky Atkins, Jud Buechler, Eric Montross, Michael Curry, Joe Smith]",3,0,3,1,3,60.000000,home,Chucky Atkins | Jud Buechler | Eric Montross | Michael Curry | Joe Smith,"(1088, 146, 376, 688, 693)",1610612765,Detroit Pistons,0.010874
1,"('1000', '1508', '165', '1749', '672')","[Shandon Anderson, Maurice Taylor, Hakeem Olajuwon, Cuttino Mobley, Matt Bullard]",2,0,2,1,2,60.000000,home,Shandon Anderson | Maurice Taylor | Hakeem Olajuwon | Cuttino Mobley | Matt Bullard,"(1000, 1508, 165, 1749, 672)",1610612745,Houston Rockets,0.005968
2,"('1732', '1888', '393', '436', '63')","[Felipe Lopez, Richard Hamilton, Rod Strickland, Juwan Howard, Michael Smith]",2,0,2,1,2,60.000000,home,Felipe Lopez | Richard Hamilton | Rod Strickland | Juwan Howard | Michael Smith,"(1732, 1888, 393, 436, 63)",1610612764,Washington Wizards,0.009665
3,"('1717', '363', '458', '919', '93')","[Dirk Nowitzki, Christian Laettner, Howard Eisley, Loy Vaught, Hubert Davis]",6,0,7,1,6,51.428571,home,Dirk Nowitzki | Christian Laettner | Howard Eisley | Loy Vaught | Hubert Davis,"(1717, 363, 458, 919, 93)",1610612742,Dallas Mavericks,-0.011417
4,"('1499', '1559', '179', '2040', '677')","[Tony Battie, Adrian Griffin, Bryant Stith, Jerome Moiso, Eric Williams]",5,0,6,1,5,50.000000,home,Tony Battie | Adrian Griffin | Bryant Stith | Jerome Moiso | Eric Williams,"(1499, 1559, 179, 2040, 677)",1610612738,Boston Celtics,-0.015858


## Build recommendation scoring table

The first recommender version combines:

- historical lineup performance
- average player impact
- lineup reliability through total duration and stint count

This produces a practical ranking score for observed lineups.

In [29]:
candidate_lineups["duration_score"] = (
    candidate_lineups["total_duration"] / candidate_lineups["total_duration"].max()
)

candidate_lineups["stint_score"] = (
    candidate_lineups["total_stints"] / candidate_lineups["total_stints"].max()
)

candidate_lineups["recommendation_score"] = (
    candidate_lineups["net_per_60"]
    + candidate_lineups["avg_player_impact"]
    + candidate_lineups["duration_score"]
    + candidate_lineups["stint_score"]
)

recommendation_lineups = candidate_lineups.sort_values(
    "recommendation_score",
    ascending=False
).reset_index(drop=True)

recommendation_lineups[[
    "team_name",
    "lineup_display",
    "net_per_60",
    "avg_player_impact",
    "total_duration",
    "total_stints",
    "recommendation_score"
]].head(10)

,team_name,lineup_display,net_per_60,avg_player_impact,total_duration,total_stints,recommendation_score
0,Detroit Pistons,Chucky Atkins | Jud Buechler | Eric Montross | Michael Curry | Joe Smith,60.000000,0.010874,3,1,60.110423
1,Washington Wizards,Felipe Lopez | Richard Hamilton | Rod Strickland | Juwan Howard | Michael Smith,60.000000,0.009665,2,1,60.103809
2,Houston Rockets,Shandon Anderson | Maurice Taylor | Hakeem Olajuwon | Cuttino Mobley | Matt Bullard,60.000000,0.005968,2,1,60.100112
3,Dallas Mavericks,Dirk Nowitzki | Christian Laettner | Howard Eisley | Loy Vaught | Hubert Davis,51.428571,-0.011417,7,1,51.538325
4,Boston Celtics,Tony Battie | Adrian Griffin | Bryant Stith | Jerome Moiso | Eric Williams,50.000000,-0.015858,6,1,50.099907
5,Phoenix Suns,Shawn Marion | Chris Dudley | Daniel Santiago | Ruben Garces | Paul McPherson,48.000000,0.018546,5,1,48.128906
6,Milwaukee Bucks,Tim Thomas | Lindsey Hunter | Glenn Robinson | Ervin Johnson | Ray Allen,45.000000,-0.002504,4,1,45.102451
7,Phoenix Suns,Shawn Marion | Chris Dudley | Ruben Garces | Paul McPherson | Rodney Rogers,42.857143,0.019693,7,1,42.998007
8,Phoenix Suns,Shawn Marion | Chris Dudley | Paul McPherson | Clifford Robinson | Corie Blount,40.000000,0.025308,6,1,40.141074
9,Dallas Mavericks,Dirk Nowitzki | Christian Laettner | Howard Eisley | Michael Finley | Shawn Bradley,40.000000,-0.012301,12,1,40.135897


##  Build opponent lineup profile helper dataset

The app will use player impact values to estimate the strength of a selected opponent lineup.

In [30]:
opponent_player_profiles = player_lookup.merge(
    player_impact[["player_id", "impact_per_60_sec"]],
    on="player_id",
    how="left"
)

opponent_player_profiles["impact_per_60_sec"] = (
    opponent_player_profiles["impact_per_60_sec"]
    .fillna(0)
)

opponent_player_profiles.head()

,player_id,player_name,impact_per_60_sec
0,920,A.C. Green,0.009566
1,2062,A.J. Guyton,-0.033552
2,243,Aaron McKie,0.014038
3,1425,Aaron Williams,0.014409
4,228,Adam Keefe,-0.027599


## Save matchup recommendation datasets

The final files are saved for the Streamlit app.

In [31]:
teams.to_csv("../data/processed/matchup_teams.csv", index=False)
team_rosters.to_csv("../data/processed/matchup_team_rosters.csv", index=False)
player_lookup.to_csv("../data/processed/matchup_player_lookup.csv", index=False)
recommendation_lineups.to_csv("../data/processed/matchup_recommendation_lineups.csv", index=False)
opponent_player_profiles.to_csv("../data/processed/matchup_opponent_player_profiles.csv", index=False)

print("Matchup recommendation datasets saved successfully.")

Matchup recommendation datasets saved successfully.


## Key findings

This notebook creates the data foundation for a matchup-based lineup recommender.

The recommendation layer uses observed historical lineups rather than generating all possible combinations. This makes the first version more reliable and realistic because recommended lineups have already appeared in the reconstructed game sample.

The recommender combines lineup performance, player impact, playing-time reliability, and stint frequency into a single recommendation score.

## Next step

The next step is to update the Streamlit dashboard with a Matchup Recommender tab.

The app will allow a user to select a team, select an opponent lineup, and receive ranked lineup recommendations based on the prepared matchup datasets.